# Week 5 — Urdu OCR: Streamlit App + Hugging Face Spaces Deployment
### SI-26 | Code Saviours Summer Internship 2026
**Project:** Urdu OCR Tool | **Week:** 5 of 5 (Final Week) | **Submitted by:** Qandeel Asim

---

### What this notebook does
This is the final step of the project: it loads the model fine-tuned in Week 4
(v8 — the full pretrained TrOCR, encoder + decoder, fine-tuned end-to-end), wraps
it in a live, testable web app (**Streamlit**, instead of Gradio), and packages
everything needed to make that app permanently public (Hugging Face Spaces).

**No training happens here.** Every cell below either *loads* something already
built in Week 4, or *uses* it. If a cell errors, the fix is almost always in
Week 4 (the saved model files), not in this notebook.

**Matches the actual Week 4 v8 architecture:** the Week 4 v8 notebook fine-tunes
the *full* pretrained `microsoft/trocr-base-printed` (its byte-level BPE decoder
can already represent Urdu via UTF-8 fallback tokens), and saves it with a single
`TrOCRProcessor` (image processor + tokenizer combined) — not a custom character
tokenizer. This notebook loads it the same way, with `TrOCRProcessor.from_pretrained()`
and `VisionEncoderDecoderModel.from_pretrained()` — no `urdu_char_vocab.pkl`, no
separate `ViTImageProcessor`, no hand-written decode-only tokenizer needed.

**Why Streamlit instead of Gradio:** Streamlit apps aren't run inline like a
Gradio `demo.launch()` cell — a Streamlit app is a separate script (`app.py`)
that runs as its own local web server (`streamlit run app.py`) on port `8501`.
Since Colab can't show that server directly, Section 7 below writes `app.py`,
starts it in the background, and opens a public tunnel to it (via `localtunnel`)
so you can click through and test it — exactly like Gradio's `share=True`, just
with one extra step (a one-time "tunnel password" = your Colab machine's IP,
which the cell prints for you).

### How to run this notebook
Run Section 0, restart the runtime, then run every remaining cell top to bottom,
in order, once. Do not skip Section 2 (Drive mount), and do not stop/interrupt
Section 7 (the Streamlit launch cell) while you're actively testing the app in
its browser tab — interrupting it kills the live server mid-request, which is
the single most common cause of "nothing happens after I click Submit."

### Sections
| # | Section | Purpose |
|---|---------|---------|
| 0 | Install everything | one-time setup (pinned to match Week 4's environment) |
| 1 | GPU check | confirms Colab gave you a GPU (optional for inference) |
| 2 | Mount Drive & locate model | finds your Week 4 v8 output automatically |
| 3 | Load the model + processor | loads the fine-tuned weights and tokenizer together |
| 4 | Inference function | the core `predict(image) -> text` logic |
| 5 | Sanity check | proves the model works *before* touching Streamlit |
| 6 | Build sample gallery | a few real test images for one-click testing |
| 7 | Launch the Streamlit app | the live, shareable demo (via a public tunnel) |
| 8 | Package for Hugging Face Spaces | app.py + requirements.txt + model, zipped |
| 9 | Generate README draft | pre-filled with your actual Week 4 v8 results |
| 10 | Submission checklist | what to hand in |

## Section 0 — Install Everything

In [1]:
# Pinned to match the Week 4 v8 environment (transformers==4.46.3), since
# a fresh Colab runtime can otherwise install a newer transformers build that
# breaks VisionEncoderDecoderModel's import path the same way Week 4 hit in
# earlier versions (ImportError: find_pruneable_heads_and_indices).
#
# streamlit replaces gradio here. We also install localtunnel (a Node package)
# so Section 7 can expose the local Streamlit server (port 8501) with a public URL,
# the same way Gradio's share=True used to.
!pip install -q streamlit "transformers==4.46.3" sentencepiece "protobuf>=3.20.2,<6"
!npm install -g localtunnel -q
print("Install complete. Now go to: Runtime > Restart session. Then run Section 1 onward.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 59.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
added 22 packages in 2s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.2
npm notice Changelog: https://github.com/npm/cli/release

## Section 1 — GPU Check

Inference works on CPU too, just slower per image (a few seconds instead of
under one). A GPU is nice to have here but not required, unlike in Week 4's
training.

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU — inference will still work, just a bit slower per image.")

CUDA available: True
GPU: Tesla T4


## Section 2 — Mount Drive and Locate the Fine-Tuned Model

Two things happen here:
1. **Find the model.** Rather than hardcoding a path (which breaks the moment
   your Drive structure changes even slightly), this walks your entire Drive
   looking for a folder that contains the files a `VisionEncoderDecoderModel`
   + `TrOCRProcessor` save together (`config.json`, `preprocessor_config.json`,
   `tokenizer_config.json`) — the ones saved at the end of Week 4 v8, Section 16
   (`trocr-urdu-finetuned-v7`). It checks that exact folder name first, then
   falls back to searching by file signature in case you renamed it.
2. **Define `resolve_image_path()`.** Week 3 and Week 4 both hit the same
   recurring issue: image paths inside `labels.csv` don't always match exactly
   where the files ended up on Drive (extra nested folders from how Drive Sync
   works). This helper tries the path as-is first, and falls back to searching
   all of Drive by filename. We define it once here and reuse it in Sections 5
   and 6.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os

MODEL_FOLDER_HINT = "trocr-urdu-finetuned-v7"  # saved at the end of Week 4 v8, Section 16

def looks_like_trocr_model_dir(files):
    """A folder saved via model.save_pretrained() + processor.save_pretrained()
    for a VisionEncoderDecoderModel + TrOCRProcessor has these together."""
    required = {"config.json", "preprocessor_config.json", "tokenizer_config.json"}
    return required.issubset(set(files))

found_model_dir = None

# Prefer the known Week 4 v8 folder name if it's there.
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if os.path.basename(root) == MODEL_FOLDER_HINT and looks_like_trocr_model_dir(files):
        found_model_dir = root
        break

# Fall back: any folder anywhere on Drive with the right file signature.
if found_model_dir is None:
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if looks_like_trocr_model_dir(files):
            found_model_dir = root
            break

if found_model_dir is None:
    raise FileNotFoundError(
        "Could not find a fine-tuned TrOCR model folder (with config.json, "
        "preprocessor_config.json, tokenizer_config.json) anywhere in My Drive. "
        "Make sure Week 4 v8, Section 16 (Save Fine-Tuned Model to Drive) was "
        "run and has fully synced to Drive before running this notebook."
    )

MODEL_DIR = found_model_dir
PROJECT_DIR = os.path.dirname(MODEL_DIR)
print("Found fine-tuned model at:", MODEL_DIR)
print("Contents:", os.listdir(MODEL_DIR))


def resolve_image_path(image_path, project_dir):
    """Resolve an image_path from labels.csv to a real file on Drive, even if the
    folder structure shifted slightly. Tries the path as given first, then falls
    back to a filename search across all of Drive."""
    candidate = os.path.join(project_dir, image_path)
    if os.path.exists(candidate):
        return candidate

    filename = os.path.basename(image_path)
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if filename in files:
            return os.path.join(root, filename)
    return None

Mounted at /content/drive
Found fine-tuned model at: /content/drive/MyDrive/Urdu_OCR_Project_v2/trocr-urdu-finetuned-v7
Contents: ['config.json', 'generation_config.json', 'model.safetensors', 'preprocessor_config.json', 'merges.txt', 'vocab.json', 'tokenizer_config.json', 'special_tokens_map.json']


## Section 3 — Load the Fine-Tuned Model + Processor

`TrOCRProcessor` bundles the image processor *and* the tokenizer in one
object — this is the same processor Week 4 v8 fine-tuned with, so it must be
loaded from `MODEL_DIR`, not re-created from the base `microsoft/trocr-base-printed`
checkpoint.

`MAX_LENGTH` also needs to match what Week 4 v8 actually used, which was
computed dynamically from the real label lengths (not a fixed guess). This
recomputes it the same way if `labels.csv` is still on Drive, and otherwise
falls back to the value from the last Week 4 v8 run.

In [4]:
from transformers import VisionEncoderDecoderModel, TrOCRProcessor
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

processor = TrOCRProcessor.from_pretrained(MODEL_DIR)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_DIR).to(device)
model.eval()  # inference mode: disables dropout, etc. — always do this before predicting

labels_path = os.path.join(PROJECT_DIR, "labels.csv")
if os.path.exists(labels_path):
    _df_for_length = pd.read_csv(labels_path)
    _lengths = [len(processor.tokenizer(str(t)).input_ids) for t in _df_for_length["text"].tolist()]
    MAX_LENGTH = max(64, max(_lengths) + 8)
else:
    MAX_LENGTH = 319  # from the last Week 4 v8 run's Section 6 output — update if your dataset changed

print("Model + processor loaded on:", device)
print("MAX_LENGTH:", MAX_LENGTH)

Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "qkv_bias": false,
  "transformers_version": "4.46.3"
}

Config of the decoder: <class 'transformers.models.trocr.modeling_trocr.TrOCRForCausalLM'> is overwritten by shared decoder config: TrOCRConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_cross_attention": true,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classifier_dropout": 0.0,
  "cross_attention_hidden_size": 768,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder

Model + processor loaded on: cuda
MAX_LENGTH: 319


## Section 4 — The Inference Function

In [5]:
from PIL import Image

@torch.no_grad()
def predict(image):
    """
    Extract Urdu text from an image using the fine-tuned TrOCR model.

    Parameters
    ----------
    image : PIL.Image.Image
        Any image containing Urdu text (photo, scan, screenshot).

    Returns
    -------
    str
        The predicted Urdu text, or "" if no image was given.
    """
    if image is None:
        return ""

    image = image.convert("RGB")
    pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)
    generated_ids = model.generate(pixel_values, max_length=MAX_LENGTH, num_beams=4)
    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return text

print("predict() is defined and ready.")

predict() is defined and ready.


## Section 5 — Sanity Check on Real Test Images

In [6]:
labels_path = os.path.join(PROJECT_DIR, "labels.csv")

if not os.path.exists(labels_path):
    print(f"labels.csv not found at {labels_path} — skipping sanity check.")
    print("This does NOT affect the Streamlit app below; it only skips this one check.")
else:
    df = pd.read_csv(labels_path)
    image_col = "image_path" if "image_path" in df.columns else df.columns[0]

    checked, matched = 0, 0
    for idx in range(min(3, len(df))):
        row = df.iloc[idx]
        resolved_path = resolve_image_path(row[image_col], PROJECT_DIR)
        checked += 1

        if resolved_path is None:
            print(f"[{idx}] Could not locate image on Drive: {row[image_col]} — skipped.")
            continue

        sample_image = Image.open(resolved_path)
        prediction = predict(sample_image)
        matched += 1

        print(f"--- Sample {idx} ---")
        print("Ground truth:", row["text"])
        print("Model output:", prediction)
        print()

    if matched == 0:
        print("No sample images could be located — this is a Drive path issue only, "
              "not a model issue. The Streamlit app below is unaffected since it uses "
              "live-uploaded images, not labels.csv.")
    else:
        print(f"Sanity check complete: {matched}/{checked} samples predicted successfully.")
        print("Note: this model's measured test accuracy is ~48% (Week 4 v8), so exact "
              "matches aren't expected on every sample — partial/near matches are normal.")

--- Sample 0 ---
Ground truth: مرثیہ اور نوحہ لکھنے میں شہرت یافتہ شعرا میر ببر علی انیس اور مرزا سلامت علی دبیر ہیں
Model output: مرثیہ اور نوحی لکھنے میں شہرت یافتہ شعرا م�ر ببر ہی، انیس اردر مرزا سلامت علی دبیر کیدہ

--- Sample 1 ---
Ground truth: ہندوستان میں ہندی سے اردو اتنی نہیں بدل گئی جتنی پاکستان میں ہے
Model output: ساتواہن سلاطین کیا صدر اور بدھ مت قبول ہے جتنی بولی اپنگريز میں گئے

--- Sample 2 ---
Ground truth: شاہی قلعہ لاہور اور شالیمار باغ لاہور
Model output: شاہی قلعہ ادار کی تعلیم بنایا ڈرام

Sanity check complete: 3/3 samples predicted successfully.
Note: this model's measured test accuracy is ~48% (Week 4 v8), so exact matches aren't expected on every sample — partial/near matches are normal.


## Section 6 — Build a Sample Gallery for One-Click Testing

In [7]:
example_image_paths = []

if os.path.exists(labels_path):
    df = pd.read_csv(labels_path)
    image_col = "image_path" if "image_path" in df.columns else df.columns[0]

    for idx in range(len(df)):
        if len(example_image_paths) >= 4:
            break
        row = df.iloc[idx]
        resolved_path = resolve_image_path(row[image_col], PROJECT_DIR)
        if resolved_path is not None:
            example_image_paths.append(resolved_path)

print(f"Found {len(example_image_paths)} example image(s) for the Streamlit gallery:")
for p in example_image_paths:
    print(" -", p)

if not example_image_paths:
    print("No example images found — the app below will still work fine with manual "
          "uploads, it just won't show one-click examples.")

Found 4 example image(s) for the Streamlit gallery:
 - /content/drive/MyDrive/Urdu_OCR_Project_v2/data/raw/augmented/urdu_0406_aug1_blur.png
 - /content/drive/MyDrive/Urdu_OCR_Project_v2/data/raw/augmented/urdu_0363_aug1_blur.png
 - /content/drive/MyDrive/Urdu_OCR_Project_v2/data/raw/augmented/urdu_0156_aug1_rotation.png
 - /content/drive/MyDrive/Urdu_OCR_Project_v2/data/raw/synthetic/urdu_0292.png


## Section 7 — Build and Launch the Streamlit App

Streamlit apps run as a standalone script, not inline in a cell like Gradio's
`demo.launch()`. This section:
1. Writes `app.py` to disk — it re-loads the model itself (Streamlit scripts run
   in their own process), copies example images into a local folder it can read,
   and builds the upload + predict UI with `st.file_uploader` / `st.button`.
2. Starts `streamlit run app.py` in the background on port `8501`.
3. Opens a public tunnel to that port with `localtunnel`, and prints the URL to
   click plus the one-time **Tunnel Password** (your Colab machine's public IP —
   the tunnel page will ask you to paste this once).

**After the link opens:** upload an image (or pick one of the examples in the
sidebar), then click **Extract Text**. Let it finish before closing the tab or
interrupting the cell below.

In [8]:
import shutil

APP_DIR = "/content/streamlit_app"
os.makedirs(APP_DIR, exist_ok=True)

# Copy example images next to app.py so the standalone Streamlit process can read them
# without needing Drive access (Drive is only mounted in this notebook's own process).
EXAMPLES_DIR = os.path.join(APP_DIR, "examples")
os.makedirs(EXAMPLES_DIR, exist_ok=True)
local_example_paths = []
for i, p in enumerate(example_image_paths):
    dest = os.path.join(EXAMPLES_DIR, f"example_{i}{os.path.splitext(p)[1]}")
    shutil.copyfile(p, dest)
    local_example_paths.append(os.path.basename(dest))

MODEL_FOLDER_NAME = os.path.basename(MODEL_DIR)
model_dest = os.path.join(APP_DIR, MODEL_FOLDER_NAME)
if not os.path.exists(model_dest):
    shutil.copytree(MODEL_DIR, model_dest)

app_py = f'''import os
import streamlit as st
import torch
from PIL import Image
from transformers import VisionEncoderDecoderModel, TrOCRProcessor

MODEL_DIR = os.path.join(os.path.dirname(__file__), "{MODEL_FOLDER_NAME}")
MAX_LENGTH = {MAX_LENGTH}
EXAMPLES_DIR = os.path.join(os.path.dirname(__file__), "examples")
EXAMPLE_FILES = {local_example_paths!r}

st.set_page_config(page_title="Urdu OCR - Code Saviours SI-26", layout="centered")


@st.cache_resource
def load_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    processor = TrOCRProcessor.from_pretrained(MODEL_DIR)
    model = VisionEncoderDecoderModel.from_pretrained(MODEL_DIR).to(device)
    model.eval()
    return processor, model, device


processor, model, device = load_model()


@torch.no_grad()
def predict(image):
    if image is None:
        return ""
    image = image.convert("RGB")
    pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)
    generated_ids = model.generate(pixel_values, max_length=MAX_LENGTH, num_beams=4)
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]


st.title("Urdu OCR")
st.caption("Code Saviours SI-26 | Fine-tuned TrOCR (microsoft/trocr-base-printed)")
st.write(
    "Upload a photo or scan containing Urdu text. This tool fine-tunes the full "
    "pretrained TrOCR model on a custom Urdu dataset to read the text and return "
    "it as editable Unicode."
)
st.info(
    "Trained on a small dataset (~1,000 unique images) — works best on clean, "
    "printed, single-line Urdu text."
)

st.sidebar.header("Try an example")
selected_example = None
if EXAMPLE_FILES:
    choice = st.sidebar.radio(
        "One-click examples",
        options=["(none)"] + EXAMPLE_FILES,
    )
    if choice != "(none)":
        selected_example = os.path.join(EXAMPLES_DIR, choice)
        st.sidebar.image(selected_example, use_container_width=True)
else:
    st.sidebar.write("No example images bundled with this app.")

uploaded_file = st.file_uploader(
    "Upload an image containing Urdu text", type=["png", "jpg", "jpeg", "bmp", "webp"]
)

image_to_predict = None
if uploaded_file is not None:
    image_to_predict = Image.open(uploaded_file)
    st.image(image_to_predict, caption="Uploaded image", use_container_width=True)
elif selected_example is not None:
    image_to_predict = Image.open(selected_example)

if st.button("Extract Text", type="primary", disabled=image_to_predict is None):
    with st.spinner("Reading the image..."):
        try:
            result = predict(image_to_predict)
        except Exception as e:
            st.error(f"ERROR while predicting: {{e}}")
            result = None
    if result is not None:
        if not result.strip():
            st.warning("Model returned empty text for this image — try a clearer or higher-contrast image.")
        else:
            st.subheader("Extracted Urdu Text")
            st.markdown(
                f"<div style='direction: rtl; font-size: 1.4em; text-align: right;'>{{result}}</div>",
                unsafe_allow_html=True,
            )
elif image_to_predict is None:
    st.caption("Upload an image or pick an example from the sidebar, then click Extract Text.")
'''

with open(os.path.join(APP_DIR, "app.py"), "w") as f:
    f.write(app_py)

print("app.py written to:", os.path.join(APP_DIR, "app.py"))

app.py written to: /content/streamlit_app/app.py


In [9]:
# Start Streamlit in the background, then open a public tunnel to port 8501
# (same purpose as Gradio's share=True, just as two steps instead of one).
import subprocess
import time

streamlit_proc = subprocess.Popen(
    [
        "streamlit", "run", os.path.join(APP_DIR, "app.py"),
        "--server.port", "8501",
        "--server.headless", "true",
    ],
    cwd=APP_DIR,
    stdout=open("/content/streamlit_log.txt", "w"),
    stderr=subprocess.STDOUT,
)

print("Starting Streamlit server, waiting a few seconds for it to come up...")
time.sleep(8)

print()
print("Your Tunnel Password (paste this on the localtunnel page if asked) is:")
!curl -s https://loca.lt/mytunnelpassword
print()
print()
print("Click the URL below to open the app (then paste the password above if prompted):")
!npx --yes localtunnel --port 8501

Starting Streamlit server, waiting a few seconds for it to come up...

Your Tunnel Password (paste this on the localtunnel page if asked) is:
136.85.16.208

Click the URL below to open the app (then paste the password above if prompted):
⠙⠹⠸⠼⠴your url is: https://sixty-eyes-think.loca.lt
⠙

**Stopping the server:** the cell above runs until you interrupt it (■ button)
or the runtime disconnects — that's normal, it's how the live app stays up while
you test it. If you need to restart the app after editing `app.py`, interrupt
this cell, then re-run it.

## Section 8 — Package Everything for Hugging Face Spaces

Copies the fine-tuned model folder as-is (weights, `TrOCRProcessor` config,
tokenizer files — all committed together) alongside `app.py` and
`requirements.txt`, so the Space can run standalone with no Google Drive
access.

**Note on Spaces + Streamlit:** when you create the Space on huggingface.co,
set its **Space SDK** to **Streamlit** (not Gradio/Docker) — Spaces then knows
to run `app.py` with `streamlit run` automatically, the same way it auto-runs
`demo.launch()` for a Gradio SDK Space.

In [10]:
import shutil

SPACES_DIR = "/content/spaces_upload"
os.makedirs(SPACES_DIR, exist_ok=True)

MODEL_FOLDER_NAME = os.path.basename(MODEL_DIR)  # e.g. "trocr-urdu-finetuned-v7"

# 1) Copy the fine-tuned model folder in as-is (weights, processor + tokenizer config)
model_dest = os.path.join(SPACES_DIR, MODEL_FOLDER_NAME)
if os.path.exists(model_dest):
    shutil.rmtree(model_dest)
shutil.copytree(MODEL_DIR, model_dest)

# 2) Copy example images in too, so the Spaces app can offer one-click examples
#    exactly like the Colab preview in Section 7.
examples_dest = os.path.join(SPACES_DIR, "examples")
os.makedirs(examples_dest, exist_ok=True)
local_example_paths = []
for i, p in enumerate(example_image_paths):
    dest = os.path.join(examples_dest, f"example_{i}{os.path.splitext(p)[1]}")
    shutil.copyfile(p, dest)
    local_example_paths.append(os.path.basename(dest))

# 3) requirements.txt — pinned to match the environment this model was trained/tested in
requirements_txt = """streamlit
torch
transformers==4.46.3
sentencepiece
protobuf>=3.20.2,<6
pillow
"""
with open(os.path.join(SPACES_DIR, "requirements.txt"), "w") as f:
    f.write(requirements_txt)

# 4) app.py — the same predict() + Streamlit logic as Section 7 above, adapted for Spaces:
#      - no drive.mount (Spaces has no Google Drive access at all)
#      - MODEL_DIR is a relative path, since app.py and the model folder are committed side by side
app_py = f'''import os
import streamlit as st
import torch
from PIL import Image
from transformers import VisionEncoderDecoderModel, TrOCRProcessor

MODEL_DIR = os.path.join(os.path.dirname(__file__), "{MODEL_FOLDER_NAME}")
MAX_LENGTH = {MAX_LENGTH}
EXAMPLES_DIR = os.path.join(os.path.dirname(__file__), "examples")
EXAMPLE_FILES = {local_example_paths!r}

st.set_page_config(page_title="Urdu OCR - Code Saviours SI-26", layout="centered")


@st.cache_resource
def load_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    processor = TrOCRProcessor.from_pretrained(MODEL_DIR)
    model = VisionEncoderDecoderModel.from_pretrained(MODEL_DIR).to(device)
    model.eval()
    return processor, model, device


processor, model, device = load_model()


@torch.no_grad()
def predict(image):
    if image is None:
        return ""
    image = image.convert("RGB")
    pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)
    generated_ids = model.generate(pixel_values, max_length=MAX_LENGTH, num_beams=4)
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]


st.title("Urdu OCR")
st.caption("Code Saviours SI-26 | Fine-tuned TrOCR (microsoft/trocr-base-printed)")
st.write(
    "Upload a photo or scan containing Urdu text. This tool fine-tunes the full "
    "pretrained TrOCR model on a custom Urdu dataset to read the text and return "
    "it as editable Unicode."
)

st.sidebar.header("Try an example")
selected_example = None
if EXAMPLE_FILES:
    choice = st.sidebar.radio("One-click examples", options=["(none)"] + EXAMPLE_FILES)
    if choice != "(none)":
        selected_example = os.path.join(EXAMPLES_DIR, choice)
        st.sidebar.image(selected_example, use_container_width=True)

uploaded_file = st.file_uploader(
    "Upload an image containing Urdu text", type=["png", "jpg", "jpeg", "bmp", "webp"]
)

image_to_predict = None
if uploaded_file is not None:
    image_to_predict = Image.open(uploaded_file)
    st.image(image_to_predict, caption="Uploaded image", use_container_width=True)
elif selected_example is not None:
    image_to_predict = Image.open(selected_example)

if st.button("Extract Text", type="primary", disabled=image_to_predict is None):
    with st.spinner("Reading the image..."):
        try:
            result = predict(image_to_predict)
        except Exception as e:
            st.error(f"ERROR while predicting: {{e}}")
            result = None
    if result is not None:
        if not result.strip():
            st.warning("Model returned empty text for this image.")
        else:
            st.subheader("Extracted Urdu Text")
            st.markdown(
                f"<div style='direction: rtl; font-size: 1.4em; text-align: right;'>{{result}}</div>",
                unsafe_allow_html=True,
            )
elif image_to_predict is None:
    st.caption("Upload an image or pick an example from the sidebar, then click Extract Text.")
'''

with open(os.path.join(SPACES_DIR, "app.py"), "w") as f:
    f.write(app_py)

print("Files ready in:", SPACES_DIR)
for item in sorted(os.listdir(SPACES_DIR)):
    print(" -", item)

Files ready in: /content/spaces_upload
 - app.py
 - examples
 - requirements.txt
 - trocr-urdu-finetuned-v7


### Uploading to your Space

**Easiest path (web UI, no terminal needed):**
1. Run the zip cell below, then download `spaces_upload.zip` from the Colab
   file browser (folder icon, left sidebar).
2. Unzip it on your computer.
3. Create your Space on huggingface.co with **SDK: Streamlit**.
4. On your Space page: **Files → Add file → Upload files**, then drag in
   `app.py`, `requirements.txt`, the `examples` folder, and the whole model
   folder (e.g. `trocr-urdu-finetuned-v7`).
5. Commit. Hugging Face builds the Space automatically — wait 2-3 minutes,
   then open your Space URL. It should be live.

**Alternative (git — faster for large model files):**
```bash
git clone https://huggingface.co/spaces/[your-username]/urdu-ocr-codesaviours-si26-[yourfirstname]
# copy the contents of spaces_upload/ into the cloned folder, then:
git add .
git commit -m "Deploy Urdu OCR app (Streamlit)"
git push
```
(Spaces automatically uses Git LFS for the model weight files — no extra setup needed.)

In [11]:
import shutil

zip_path = shutil.make_archive("/content/spaces_upload", "zip", SPACES_DIR)
print("Zipped to:", zip_path)
print("Download it from the Colab file browser (folder icon on the left) and upload its contents to your Space.")

Zipped to: /content/spaces_upload.zip
Download it from the Colab file browser (folder icon on the left) and upload its contents to your Space.


## Section 9 — Generate a README Draft

Builds a starting-point `README.md`, pre-filled with the actual dataset and
results facts from Weeks 1-4 v8.

**You must still edit this before submitting:**
- Replace `[your-username]` and `[yourfirstname]` with your actual Space URL.
- Replace the GitHub URL if your repo name differs.

In [12]:
readme_md = """# Urdu OCR — A Fine-Tuned TrOCR Model for Extracting Text from Urdu Images

## What problem this solves and why it matters
Optical Character Recognition for Urdu lags far behind Latin-script OCR: Urdu's
cursive, context-dependent Nastaliq script and the scarcity of labeled datasets make
off-the-shelf tools like Tesseract perform poorly out of the box. This project
fine-tunes a TrOCR model specifically on Urdu text so it can read real-world Urdu
images — for example, digitizing scanned Urdu documents, signboards, or book pages
that would otherwise have to be transcribed by hand.

## How it works
[TrOCR](https://huggingface.co/microsoft/trocr-base-printed) pairs a vision encoder
(which "looks" at the image) with a text decoder (which "writes out" what it reads),
trained end-to-end on paired image/text data. Its decoder's tokenizer is a byte-level
BPE tokenizer that can already represent any Unicode text, Urdu included, via UTF-8
byte-level fallback tokens — so this project fine-tunes the *full* pretrained model
(encoder AND decoder) directly on a labeled Urdu image dataset, rather than training
a decoder from scratch, so the model only has to learn Urdu, not language modeling
from zero.

## Live demo
**[Try it here](https://huggingface.co/spaces/[your-username]/urdu-ocr-codesaviours-si26-[yourfirstname])**

Built with **Streamlit**.

## How to run it locally
```bash
git clone https://github.com/qandeelasim13/URDU-OCR-PROJECT-CODE-SAVIOURS-SI-2026-QANDEEL-ASIM.git
cd URDU-OCR-PROJECT-CODE-SAVIOURS-SI-2026-QANDEEL-ASIM
pip install -r requirements.txt
streamlit run app.py
```
Then open the local URL Streamlit prints in your terminal (usually http://localhost:8501).

## Dataset details
- labels.csv: 3,160 rows (about 65% pre-augmented rotation/blur/brightness copies
  of 1,348 unique source images), drawn from four sources: the UTRSet-Real dataset,
  synthetic images generated with the Noto Nastaliq Urdu font, augmented variants
  of those images, and manual screenshots labeled via EasyOCR.
- Mix of printed book-style and signboard-style text, in varying fonts, backgrounds,
  and image sizes.
- Leakage-safe split, grouped by parent image, so augmented copies of the same
  source image never appear in both train and test.

## Results (Week 4 v8)
- Character Error Rate (CER) on the held-out test set: **0.52**
- Character-level accuracy: **47.66%**
- Training loss: 5.00 -> 0.18 (per-epoch average, over 15 epochs, two-phase
  fine-tuning: decoder-only warm-up, then the full model unfrozen at a lower
  learning rate)

This is below what a larger dataset would likely support (the notebook's own
working estimate was 80-95% given more data) -- the main limiting factor is
dataset size (~1,000 unique images). With more time or data, the next steps
would be: growing the labeled dataset, training for more epochs, and trying
data augmentation targeted at the error patterns seen in the worst-predictions
report (Week 4 v8, Section 13).

## Credit
Qandeel Asim
Built during the Code Saviours ML/AI Internship — Batch SI-26.
"""

with open(os.path.join(SPACES_DIR, "README.md"), "w") as f:
    f.write(readme_md)

readme_drive_path = os.path.join(PROJECT_DIR, "README_week5_draft.md")
with open(readme_drive_path, "w") as f:
    f.write(readme_md)

print("README draft written to:")
print(" -", os.path.join(SPACES_DIR, "README.md"))
print(" -", readme_drive_path)
print()
print(readme_md)

README draft written to:
 - /content/spaces_upload/README.md
 - /content/drive/MyDrive/Urdu_OCR_Project_v2/README_week5_draft.md

# Urdu OCR — A Fine-Tuned TrOCR Model for Extracting Text from Urdu Images

## What problem this solves and why it matters
Optical Character Recognition for Urdu lags far behind Latin-script OCR: Urdu's
cursive, context-dependent Nastaliq script and the scarcity of labeled datasets make
off-the-shelf tools like Tesseract perform poorly out of the box. This project
fine-tunes a TrOCR model specifically on Urdu text so it can read real-world Urdu
images — for example, digitizing scanned Urdu documents, signboards, or book pages
that would otherwise have to be transcribed by hand.

## How it works
[TrOCR](https://huggingface.co/microsoft/trocr-base-printed) pairs a vision encoder
(which "looks" at the image) with a text decoder (which "writes out" what it reads),
trained end-to-end on paired image/text data. Its decoder's tokenizer is a byte-level
BPE tokenizer

## Section 10 — Submission Checklist

- [ ] Live Hugging Face Space link (open it fresh and test it before submitting)
- [ ] GitHub repo link with the complete README (no `[brackets]` left unedited)
- [ ] This Week 5 Colab notebook link
- [ ] Screenshot of the working Streamlit demo (from Section 7)

### If something still doesn't work
Run the cells in order top to bottom in a **fresh** Colab runtime
(Runtime → Restart session), without skipping any cell. Almost every "it's not
working" case comes down to either an out-of-order run or the Streamlit cell
in Section 7 being interrupted mid-request — both are fixed by a clean
top-to-bottom run.